# Gemma 4 31B Fine-Tuning on Eight Benchmark Types

This notebook fine-tunes `google/gemma-4-31B-it` on one representative benchmark for each evaluation type defined in `Datasets.tex`. It is intended to run from a clone of this repository on a RunPod B300.

The run trains one balanced LoRA adapter on the combined eight-benchmark mixture. Training and validation examples are exported from each benchmark's training split; final evaluation must use held-out splits or disjoint data.

## Selected Benchmarks

| Type | Task | Benchmark | Reason |
|---|---|---|---|
| `L` | Labeling | `fashion_mnist` | Clean ten-class classification baseline |
| `A` | Visual question answering | `docvqa` | Tests OCR, layout understanding, and visual reasoning |
| `B` | Bounding-box detection | `openimages_v4_detection` | Diverse object vocabulary with bounding boxes |
| `C` | Captioning | `mscoco_caption` | Standard general-purpose image captioning corpus |
| `E` | Editing prompt reconstruction | `magicbrush` | Real before/after image-edit pairs with instructions |
| `G` | Generation prompt reconstruction | `diffusiondb` | Generated images paired with source prompts |
| `P` | Preference | `pick_a_pic` | Human preferences between generated-image pairs |
| `R` | Rating | `tad66k` | Human aesthetic scores across diverse themes |

The exporter converts every task to the same supervised structure: image, task prompt, and exact target response.

## 1. Configuration

In [ ]:
from pathlib import Path

BENCHMARKS = {
    "L": {"benchmark": "fashion_mnist", "train_split": "train"},
    "A": {"benchmark": "docvqa", "train_split": "train"},
    "B": {"benchmark": "openimages_v4_detection", "train_split": "train"},
    "C": {"benchmark": "mscoco_caption", "train_split": "train"},
    "E": {"benchmark": "magicbrush", "train_split": "train"},
    "G": {"benchmark": "diffusiondb", "train_split": "train"},
    "P": {"benchmark": "pick_a_pic", "train_split": "train"},
    "R": {"benchmark": "tad66k", "train_split": "train"},
}

# Equal counts prevent a large source dataset from dominating the mixture.
TRAIN_EXAMPLES_PER_TYPE = 1000
VALIDATION_EXAMPLES_PER_TYPE = 200
EPOCHS = 2
LEARNING_RATE = "1e-4"
GRADIENT_ACCUMULATION_STEPS = 16
LORA_RANK = 32
SEED = 42

DATA_ROOT = Path("fine_tuning/data/eight_types")
COMBINED_DIR = DATA_ROOT / "combined"
ADAPTER_DIR = Path("fine_tuning/output/gemma-4-31b-eight-types-lora")
RUN_TRAINING = True

BENCHMARKS

## 2. Locate the Repository and Install Dependencies

Run this notebook from anywhere inside the cloned repository. The setup cell installs the repository and Gemma fine-tuning dependencies into the current Python environment.

In [ ]:
import os
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "fine_tuning").is_dir() and (candidate / "benchmarks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing fine_tuning/ and benchmarks/.")

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
print("Repository:", REPO_ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-r", "fine_tuning/requirements.txt"],
    check=True,
)

def run_command(command):
    print("$", " ".join(str(part) for part in command), flush=True)
    subprocess.run([str(part) for part in command], cwd=REPO_ROOT, check=True)

## 3. Verify the B300 and Hugging Face Access

Accept the Gemma model license before continuing: <https://huggingface.co/google/gemma-4-31B-it>.

In [ ]:
import getpass
import torch

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ").strip()
if not os.environ["HF_TOKEN"]:
    raise RuntimeError("HF_TOKEN is required for the Gemma checkpoint and some datasets.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Start this notebook on a GPU RunPod.")

properties = torch.cuda.get_device_properties(0)
print("GPU:", properties.name)
print(f"VRAM: {properties.total_memory / 1024**3:.1f} GiB")
print("BF16 supported:", torch.cuda.is_bf16_supported())
run_command([sys.executable, "fine-tuning/prepare_benchmark.py", "--help"])
run_command([sys.executable, "fine_tuning/train_gemma4_31b.py", "--help"])

## 4. Export Balanced Training and Validation Slices

For each type, the exporter requests one combined slice from the named training split and then divides it into disjoint training and validation portions.

In [ ]:
for task_type, spec in BENCHMARKS.items():
    benchmark_dir = DATA_ROOT / f"{task_type}_{spec['benchmark']}"
    train_manifest = benchmark_dir / "train_manifest.jsonl"
    validation_manifest = benchmark_dir / "validation_manifest.jsonl"
    if train_manifest.exists() and validation_manifest.exists():
        print(f"{task_type}: reusing {benchmark_dir}")
        continue

    run_command([
        sys.executable,
        "fine-tuning/prepare_benchmark.py",
        "--benchmark", spec["benchmark"],
        "--train-split", spec["train_split"],
        "--train-examples", TRAIN_EXAMPLES_PER_TYPE,
        "--validation-examples", VALIDATION_EXAMPLES_PER_TYPE,
        "--output-dir", benchmark_dir,
    ])

## 5. Merge and Shuffle the Eight Types

Images are copied into type-specific subdirectories so filenames cannot collide. The combined manifests remain balanced by construction.

In [ ]:
import json
import random
import shutil

def merge_split(split: str) -> Path:
    records = []
    image_root = COMBINED_DIR / "images"
    image_root.mkdir(parents=True, exist_ok=True)

    for task_type, spec in BENCHMARKS.items():
        source_dir = DATA_ROOT / f"{task_type}_{spec['benchmark']}"
        source_manifest = source_dir / f"{split}_manifest.jsonl"
        type_image_dir = image_root / task_type
        type_image_dir.mkdir(parents=True, exist_ok=True)

        for index, line in enumerate(source_manifest.read_text(encoding="utf-8").splitlines()):
            if not line.strip():
                continue
            record = json.loads(line)
            source_image = source_dir / record["image_path"]
            suffix = source_image.suffix or ".png"
            target_image = type_image_dir / f"{split}_{index:06d}{suffix}"
            if not target_image.exists():
                shutil.copy2(source_image, target_image)
            record["image_path"] = target_image.relative_to(COMBINED_DIR).as_posix()
            record["task_type"] = task_type
            record["benchmark"] = spec["benchmark"]
            records.append(record)

    random.Random(f"{SEED}:{split}").shuffle(records)
    output_path = COMBINED_DIR / f"{split}_manifest.jsonl"
    output_path.write_text(
        "".join(json.dumps(record, ensure_ascii=True) + "\n" for record in records),
        encoding="utf-8",
    )
    print(f"Wrote {len(records):,} {split} examples to {output_path}")
    return output_path

TRAIN_MANIFEST = merge_split("train")
VALIDATION_MANIFEST = merge_split("validation")

In [ ]:
from collections import Counter
from IPython.display import display
from PIL import Image

def read_manifest(path: Path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_records = read_manifest(TRAIN_MANIFEST)
validation_records = read_manifest(VALIDATION_MANIFEST)
print("Training distribution:", Counter(record["task_type"] for record in train_records))
print("Validation distribution:", Counter(record["task_type"] for record in validation_records))

for task_type in BENCHMARKS:
    example = next(record for record in train_records if record["task_type"] == task_type)
    print(f"\n{task_type} / {example['benchmark']}")
    print("Prompt:", example["prompt"][:500])
    print("Target:", example["answer"][:300])
    display(Image.open(COMBINED_DIR / example["image_path"]))

## 6. Train the Combined Gemma Adapter

The default run uses BF16 LoRA, freezes the vision tower, masks prompt tokens from the loss, and selects the checkpoint with the lowest validation loss.

In [ ]:
training_command = [
    sys.executable,
    "fine_tuning/train_gemma4_31b.py",
    "--train-manifest", TRAIN_MANIFEST,
    "--validation-manifest", VALIDATION_MANIFEST,
    "--output-dir", ADAPTER_DIR,
    "--epochs", EPOCHS,
    "--learning-rate", LEARNING_RATE,
    "--gradient-accumulation-steps", GRADIENT_ACCUMULATION_STEPS,
    "--lora-rank", LORA_RANK,
    "--seed", SEED,
]

if RUN_TRAINING:
    run_command(training_command)
else:
    print("Training skipped. Set RUN_TRAINING = True to start it.")

## 7. Check the Saved Adapter

The output is a PEFT adapter and processor configuration. Keep the `google/gemma-4-31B-it` base checkpoint available when loading it for evaluation.

In [ ]:
if not ADAPTER_DIR.exists():
    print("Adapter directory does not exist yet:", ADAPTER_DIR)
else:
    for path in sorted(ADAPTER_DIR.iterdir()):
        size_mb = path.stat().st_size / 1024**2 if path.is_file() else 0
        print(f"{path.name:40s} {size_mb:10.2f} MB")

## Notes

- Do not evaluate on examples exported into either combined manifest.
- Some datasets require Hugging Face authentication or may download large archives.
- If BF16 LoRA runs out of memory because of unusually large images, append `--quantization 4bit` to `training_command`.
- Increase `TRAIN_EXAMPLES_PER_TYPE` only after the complete eight-type pipeline succeeds.
- A balanced mixture is useful for broad adaptation, but separate per-type adapters remain a useful ablation.